# 04 — Measure RAG quality without hiding failures

**10–15 minute lab.** Follow one visible path: **QUERY → RETRIEVAL RESULTS → GENERATED ANSWER → METRICS**.

The first three checkpoints use controlled data so every score is explainable. The optional appendix runs the same flow against PostgreSQL and Ollama.

In [ ]:
from pathlib import Path

from raglab.evaluation import (
    GenerationOutput,
    RetrievalOutput,
    build_report,
    compare_reports,
    evaluate_generation,
    evaluate_retrieval,
    load_cases,
)

DATASET_PATH = Path("data/evaluation/aster_greenhouse_controller_v1.json")
for repository in (Path.cwd(), *Path.cwd().parents):
    if (repository / "pyproject.toml").is_file() and (repository / DATASET_PATH).is_file():
        break
else:
    raise FileNotFoundError(DATASET_PATH)


def section_name(source_id: str) -> str:
    return source_id.rsplit("/", 1)[-1].replace("-", " ").title()


DATASET = repository / DATASET_PATH
dataset = load_cases(DATASET)
cases = dataset.cases

## Checkpoint 1 — Objective: define what a correct answer looks like

**Run:** read each query's grading rubric before running retrieval or generation. The labels replace internal IDs with language a human can review.

In [ ]:
print(f"Dataset: {dataset.dataset_id} ({dataset.schema_version})")
for number, case in enumerate(cases, 1):
    print(f"\nCASE {number}")
    print(f"QUERY: {case.question}")
    print("EXPECTED ANSWER MUST INCLUDE:")
    for fact in case.expected_facts:
        print(f"  - {fact.text}")
        print("    ACCEPTED WORDING:", " | ".join(fact.answer_variants))
        sections = ", ".join(section_name(item) for item in fact.evidence_source_ids)
        print("    SUPPORTING MANUAL SECTIONS:", sections)
    print("KNOWN WRONG CLAIMS — must not appear; diagnostic flags, not score inputs:")
    for phrase in case.forbidden_phrases:
        print(f"  - {phrase}")

### What to observe

This is the grading rubric, not a RAG execution. **Expected facts** define required meaning, **accepted wording** allows literal paraphrases, and **supporting sections** define valid evidence. **Known wrong claims** are explicit contradictions that must not appear; V1 reports them as separate diagnostic flags rather than silently changing a metric.

### Conclusion

A score is trustworthy only when a human can first inspect what counts as correct, supported, and known to be wrong.

## Checkpoint 2 — Objective: see exactly what retrieval returns and how it is scored

**Run:** create a controlled top-3 ranking for each query. These are evidence labels chosen for the lesson—not live chunk content from PostgreSQL—so every numerator, denominator, and rank stays visible.

In [ ]:
top_k = 3
retrieval_outputs = {}
for case in cases:
    expected = case.relevant_source_ids
    third = expected[1] if len(expected) > 1 else "unrelated-maintenance-note"
    retrieval_outputs[case.id] = RetrievalOutput(
        ranked_source_ids=(expected[0], "unrelated-greenhouse-note", third)
    )

retrieval_results = evaluate_retrieval(cases, retrieval_outputs, top_k=top_k)
for case, result in zip(cases, retrieval_results, strict=True):
    print(f"\nQUERY: {case.question}")
    print("CONTROLLED RETRIEVAL RESULTS — section labels only, not live content:")
    for rank, source_id in enumerate(result.ranked_source_ids, 1):
        label = "RELEVANT" if source_id in result.relevant_source_ids else "IRRELEVANT"
        print(f"  {rank}. [{label}] {section_name(source_id)}")
    found = len(result.retrieved_relevant_source_ids)
    total = len(result.relevant_source_ids)
    first = next(
        (
            rank
            for rank, source_id in enumerate(result.ranked_source_ids, 1)
            if source_id in result.relevant_source_ids
        ),
        None,
    )
    print(
        f"FORMULA: Precision@{top_k} = {found} relevant / {top_k} results = "
        f"{result.precision_at_k:.3f}"
    )
    print(f"FORMULA: Recall@{top_k} = {found} found / {total} expected = {result.recall_at_k:.3f}")
    print(f"FORMULA: MRR@{top_k} = 1 / first relevant rank {first} = {result.mrr_at_k:.3f}")
    missed = [section_name(item) for item in result.missed_relevant_source_ids]
    print("MISSED EXPECTED SECTIONS:", ", ".join(missed) if missed else "none")

### What to observe

Each query is followed by the exact controlled ranking used by the metrics. An irrelevant result lowers precision; missing an expected section lowers recall; moving the first relevant result down lowers MRR. This checkpoint evaluates retrieval only—there is no generated answer yet.

### Conclusion

Retrieval answers “did we bring useful evidence?” It does not answer “did the LLM use that evidence correctly?”

## Checkpoint 3 — Objective: keep retrieval fixed and expose a generation regression

**Run:** use the same retrieval results twice. First score a complete supported answer; then replace only the first generated answer with a known wrong claim and an invented citation.

In [ ]:
generation_outputs = {
    case.id: GenerationOutput(
        answer=" ".join(fact.answer_variants[0] for fact in case.expected_facts),
        cited_source_ids=case.relevant_source_ids,
    )
    for case in cases
}
generation_results = evaluate_generation(cases, generation_outputs)
baseline = build_report(dataset, retrieval_results, generation_results)

changed_case = cases[0]
regressed_outputs = dict(generation_outputs)
regressed_outputs[changed_case.id] = GenerationOutput(
    answer=changed_case.forbidden_phrases[0],
    cited_source_ids=("invented-source",),
)
regressed_results = evaluate_generation(cases, regressed_outputs)
regression = build_report(dataset, retrieval_results, regressed_results)
comparison = compare_reports(baseline, regression)

before = generation_outputs[changed_case.id]
after = regressed_outputs[changed_case.id]
before_score = generation_results[0]
after_score = regressed_results[0]
print(f"QUERY: {changed_case.question}")
print("UNCHANGED RETRIEVAL INPUT — identical for baseline and regression:")
for rank, source_id in enumerate(retrieval_outputs[changed_case.id].ranked_source_ids, 1):
    label = "RELEVANT" if source_id in changed_case.relevant_source_ids else "IRRELEVANT"
    print(f"  {rank}. [{label}] {section_name(source_id)}")
print("\nBASELINE GENERATED ANSWER:")
print(before.answer)
print("BASELINE CITATIONS:", ", ".join(section_name(item) for item in before.cited_source_ids))
print("\nREGRESSED GENERATED ANSWER:")
print(after.answer)
print("REGRESSED CITATIONS:", ", ".join(section_name(item) for item in after.cited_source_ids))
print("KNOWN WRONG CLAIMS FOUND:", after_score.forbidden_phrase_hits)
print("\nGENERATION METRICS FOR THIS QUERY (baseline → regression):")
print(f"  Fact coverage: {before_score.fact_coverage:.3f} → {after_score.fact_coverage:.3f}")
print(
    "  Grounded fact coverage:",
    f"{before_score.grounded_fact_coverage:.3f} → {after_score.grounded_fact_coverage:.3f}",
)
print(
    "  Citation precision:",
    f"{before_score.citation_precision:.3f} → {after_score.citation_precision:.3f}",
)
retrieval_deltas = [item.difference for item in comparison.retrieval.values()]
generation_deltas = [item.difference for item in comparison.generation.values()]
print("\nSTAGE DIAGNOSIS")
print("Retrieval: UNCHANGED — the same ranking was reused; all metric deltas are 0.000")
print(
    "Generation: REGRESSED — only the answer and citation changed; summary deltas:",
    ", ".join(f"{value:.3f}" for value in generation_deltas),
)

### What to observe

The query and retrieval ranking are printed once because they are identical in both reports. The baseline and regressed **generated answers** are shown side by side, followed by their generation metrics. The stage diagnosis is explicit: retrieval stayed unchanged; generation regressed.

### Conclusion

Separate stage inputs make causality visible. We know this failure belongs to generation because the experiment changed only the answer and citation.

## Optional appendix — live PostgreSQL and Ollama evaluation

Run the first cell below to assign `RAGLAB_RUN_EVALUATION_NOTEBOOK=1` explicitly before the live cell. Automated hermetic tests neutralize this activation only in their in-memory notebook copy. Index the Aster manual and set `RAGLAB_EVALUATION_COLLECTION` when the collection is not `documents`.

Unlike the controlled checkpoints, the live flow calls the reusable evaluation application and prints the actual query, readable retrieved chunks, full generated answer, citations, and then a concise stage summary.


In [ ]:
import os

os.environ["RAGLAB_RUN_EVALUATION_NOTEBOOK"] = "1"
os.environ.setdefault("RAGLAB_EVALUATION_COLLECTION", "greenhouse-manuals")


In [ ]:
import os

if os.environ.get("RAGLAB_RUN_EVALUATION_NOTEBOOK") == "1":
    from raglab.evaluation_application import (
        LiveEvaluationSettings,
        create_live_evaluation_application,
    )
    from raglab.retrieval import RetrievalConfig

    def citation_label(citation):
        heading = " > ".join(citation.heading_path) or citation.title or "No heading"
        return f"{citation.source_name} — {heading}"

    def readable_excerpt(content: str, limit: int = 500) -> str:
        compact = " ".join(content.split())
        return compact if len(compact) <= limit else compact[: limit - 1] + "…"

    collection = os.environ.get("RAGLAB_EVALUATION_COLLECTION", "documents")
    settings = LiveEvaluationSettings.from_env()
    application = create_live_evaluation_application(
        settings,
        retrieval_config=RetrievalConfig(top_k=top_k, candidate_k=max(50, top_k)),
        generation_config=settings.generation_config(minimum_sources=1),
    )
    live_run = application.run(dataset, collection=collection)
    for trace in live_run.traces:
        case = trace.case
        generated = trace.response
        retrieved = generated.retrieval
        print(f"\nQUERY: {case.question}")
        print("LIVE RETRIEVAL RESULTS — actual content passed to generation:")
        for rank, result in enumerate(retrieved.results, 1):
            print(f"  {rank}. {citation_label(result.citation)}")
            print(f"     {readable_excerpt(result.content)}")
        sources = {source.id: source for source in generated.sources}
        cited_ids = generated.source_ids or tuple(sources)
        selected = [citation_label(sources[item].citation) for item in cited_ids]
        print("SELECTED EVIDENCE:", "; ".join(selected) if selected else "none")
        metrics = generated.metrics
        print(
            f"SELECTION: calls={metrics.selection_calls}, "
            f"invalid quotes={metrics.facts_invalid_quotes}, "
            f"NLI rejected={metrics.facts_nli_rejected}"
        )
        print("GENERATED ANSWER:")
        print(generated.answer)
        print("CITATIONS:")
        for source_id in cited_ids:
            print("  -", citation_label(sources[source_id].citation))
        if not cited_ids:
            print("  - none")

    live_report = live_run.report
    print("\nLIVE STAGE SUMMARY")
    print(
        "Retrieval:",
        ", ".join(f"{name}={value:.3f}" for name, value in live_report.retrieval_summary.items()),
    )
    print(
        "Generation:",
        ", ".join(f"{name}={value:.3f}" for name, value in live_report.generation_summary.items()),
    )
else:
    print("Live evaluation disabled; set RAGLAB_RUN_EVALUATION_NOTEBOOK=1 to enable it.")
